# Book 1 — Data Exploration, Cleaning, and Analysis

**"Readers also read…": a Goodreads book recommender** · SIADS 593 Milestone I, Fall 2026
David Hook · Olivia Cain · Sarah Ehlen

This is the first of three notebooks. It downloads nothing itself; it reads the raw files listed
in Section 1, decides what a *book* and a *like* mean for this project, draws the working sample
of readers, holds some readers out for testing, joins the catalogue to Open Library, and writes
the files that **Book 2 (Modeling and Model Evaluation)** and **Book 3 (Ranked Fusion Recommender)**
use. Each decision is stated where it is made, next to the evidence for it.

| Section | What happens | Output |
|---|---|---|
| 1 | Where the raw files come from and how to place them | — |
| 2 | UCSD book metadata becomes one row per *work* | `data/processed/book_dimension.pkl` |
| 3 | UCSD interactions become a 50,000-reader sample at work level | `data/processed/interactions_sample_50k.pkl` |
| 4 | Readers are split into train, validation and test | `data/processed/user_splits.pkl` |
| 5 | How much evidence each book has, and the minimum-likes decision | `data/processed/train_likes_per_work.pkl` |
| 6 | Open Library adds subjects and current ratings through a Goodreads-ID join | `data/processed/open_library_match.pkl` |
| 7 | What Book 2 receives | data dictionary |
| 8 | Visualization 1: the interactive evidence explorer | — |

**How the notebook runs.** The code lives in `scripts/*.py`; each cell here calls one or two
functions and shows the result. Every expensive step saves a `.pkl` file and, on later runs,
loads it instead of recomputing. The first full run takes about eight minutes (one to two for the
book dimension, one for the reader sample, four for the Open Library scan); later runs take
under two.
Set a `REBUILD_*` flag in the setup cell to force a step to run again.

**Seed.** Every random choice in the project uses `SEED = 593`. Re-running the notebook draws the
same readers and assigns them to the same splits.

In [1]:
# Setup: paths, flags and the project modules this notebook narrates.
import sys
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

# Charts are emitted as Vega-Lite JSON, which Jupyter and VS Code render with their built-in
# renderer; the default HTML output needs scripts from a CDN and stays blank in VS Code.
alt.renderers.enable("mimetype")

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == "book1" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT / "scripts"))

import data_sources as ds            # Section 1: download URLs and the file check
import book_dimension as bd          # Section 2: raw metadata -> one row per work
import sample_interactions as si     # Section 3: 229M interactions -> 50,000-reader sample
import train_test_split as tts       # Section 4: hash-based split by reader
import evidence_explorer as ex       # Section 5: Visualization 1
import catalogue_charts as cc        # Sections 2 and 6: the static charts
import open_library as ol            # Section 6: Sarah's lookup and the Open Library dumps

RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
OPEN_LIBRARY = RAW / "openlibrary"
PROCESSED.mkdir(parents=True, exist_ok=True)

SEED = 593                            # the course number; used for the reader sample and the split
SAMPLE_READERS = 50_000
MIN_LIKES = 10                        # the recommendable floor decided in Section 5

REBUILD_BOOK_DIMENSION = False        # True re-runs the build from raw even if the pickle exists
REBUILD_INTERACTIONS = False          # True redraws the sample from the 4.3 GB CSV
RESCAN_OPEN_LIBRARY = False           # True re-scans the 12.6 GB editions dump

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 200)

## 1 · The data and where to get it

The recommender needs two kinds of information: **who shelved and rated which books**, and
**what each book is**, so results can be shown, filtered and judged. We use three sources. Two
are bulk downloads from UCSD's Goodreads collection (scraped in 2017); the third is Open Library,
which we reach both through its API (Section 6.1) and through its monthly data dumps (Section 6.2).

### 1.1 Download instructions

Download each file below and save it at the path shown, relative to the project folder. The
UCSD files are linked from the dataset page (`cseweb.ucsd.edu/~jmcauley/datasets/goodreads.html`).
Open Library's `..._latest` links redirect to the current month's file; we used the dump dated
**2026-09-30** and keep that date in the file name so a later dump does not change the join
without anyone noticing. Total size is about 23 GB; none of it is included in the submission.

In [2]:
# The download list. Save each URL to the path in the first column. (Section 1.3 describes the contents.)
ds.download_table()[["download from", "size (GB)"]]

,download from,size (GB)
save as,,
data/raw/goodreads_interactions.csv,https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions.csv,4.32
data/raw/book_id_map.csv,https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/book_id_map.csv,0.04
data/raw/user_id_map.csv,https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/user_id_map.csv,0.03
data/raw/goodreads_books.json.gz,https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz,2.08
data/raw/goodreads_book_genres_initial.json.gz,https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_book_genres_initial.json.gz,0.02
data/raw/goodreads_book_authors.json.gz,https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_book_authors.json.gz,0.02
data/raw/goodreads_book_series.json.gz,https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_book_series.json.gz,0.03
data/raw/openlibrary/ol_dump_editions_2026-09-30.txt.gz,https://openlibrary.org/data/ol_dump_editions_latest.txt.gz,12.62
data/raw/openlibrary/ol_dump_works_2026-09-30.txt.gz,https://openlibrary.org/data/ol_dump_works_latest.txt.gz,4.07


### 1.2 Checking what is in place

The cell below reports which raw files are present and whether each one has its full size. A
partial download of a gzip file reads without error up to the point it was cut, so the size is
the reliable check.

In [3]:
# Which raw files are present and complete.
ds.inventory(PROJECT)[["present", "size on disk (GB)", "expected (GB)", "complete"]]

,present,size on disk (GB),expected (GB),complete
file,,,,
data/raw/goodreads_interactions.csv,True,4.319,4.319,True
data/raw/book_id_map.csv,True,0.038,0.038,True
data/raw/user_id_map.csv,True,0.035,0.035,True
data/raw/goodreads_books.json.gz,True,2.083,2.083,True
data/raw/goodreads_book_genres_initial.json.gz,True,0.024,0.024,True
data/raw/goodreads_book_authors.json.gz,True,0.018,0.018,True
data/raw/goodreads_book_series.json.gz,True,0.028,0.028,True
data/raw/openlibrary/ol_dump_editions_2026-09-30.txt.gz,True,12.617,12.617,True
data/raw/openlibrary/ol_dump_works_2026-09-30.txt.gz,True,4.073,4.073,True


### 1.3 What each source contains

| Source | Files | Format | Records | Key variables |
|---|---|---|---|---|
| UCSD **interactions** | `goodreads_interactions.csv` with `book_id_map.csv` and `user_id_map.csv` | CSV | 228,648,342 rows; 876,145 readers; 2,360,650 editions | `user_id`, `book_id` (an edition index), `is_read`, `rating` 0–5 where 0 means "not rated" |
| UCSD **book metadata** | `goodreads_books.json.gz` plus genres, authors, series | gzipped JSON lines | 2,360,655 editions belonging to 1,521,963 works | `book_id`, `work_id`, title, authors, `ratings_count`, `average_rating`, ISBNs, year, language, 10 genre weights |
| **Open Library** | API endpoints `search.json`, `/isbn/{isbn}.json`, `/works/{key}.json`; dumps of editions, works and ratings | JSON; tab-separated dumps | tens of millions of editions; 709,550 works with ratings | `identifiers.goodreads`, work key, `subjects`, star ratings |

The interactions file is the primary source. The metadata is what tells us which editions are
the same book, so it is needed to clean the interactions, not only to describe them. Open Library
is the second source and the second access method the project requires.

## 2 · UCSD book metadata: from editions to works

### 2.1 The raw file

The metadata file has one row per **edition**: each printing, format or translation of a book.
Every number is stored as a string, missing values are empty strings, and `authors`, `series`,
`popular_shelves` and `similar_books` are nested lists. The first record shows all of this.

In [4]:
# The first raw edition record, before any cleaning.
first_record = bd.read_first_record(RAW / "goodreads_books.json.gz")
pd.Series(first_record).to_frame("value").head(14)   # 14 of its 29 fields; the rest follow the same pattern

,value
isbn,0312853122
text_reviews_count,1
series,[]
country_code,US
language_code,
popular_shelves,"[{'count': '3', 'name': 'to-read'}, {'count': '1', 'name': 'p'}, {'count': '1', 'name': 'collection'}, {'count': '1', 'name': 'w-c-fields'}, {'count': '1', 'name': 'biography'}]"
asin,
is_ebook,false
average_rating,4.00
kindle_asin,


### 2.2 Decision: a book is a *work*

*Killing Floor* appears 50 times in the raw file: mass market, hardcover, audiobook, Audio CD,
Dwarsligger, plus translations. If each edition stayed a separate item, readers of different
editions would never appear to share a book, and a recommender would mostly return other editions
of the book the reader named. We therefore combine editions under the `work_id` Goodreads already
assigns. The rules:

- **The most-rated edition is the canonical one** and supplies the descriptive fields (title, year,
  pages, language, ISBNs), so each row describes a real printing. Ties go to the lowest `book_id`.
- **`ratings_count` and `text_reviews_count` are summed** across editions. **`avg_rating` is
  recomputed** as the ratings-weighted mean.
- **`n_shelves` and `n_similar` are not summed**, because the source caps them per edition (100 and
  18), so a sum would not measure anything.
- **English language codes become `eng`.** `en`, `en-US`, `en-GB`, `en-CA` and `en-IN` are one
  language. Filtering on `eng` alone would have dropped 157,649 English books. `enm` (Middle English)
  is left as is.
- **Seven columns are dropped**: `popular_shelves` and `description` (three quarters of the file;
  kept as `n_shelves` and `desc_len`), `publisher`, `format`, `url`, `asin`, `kindle_asin`, and the
  day/month parts of the publication date.
- **Years keep their sign.** 354 works have BC dates (the *Iliad* is -750). Clamping to zero would
  have deleted them.
- **Nothing is filtered.** Rows disappear only by merging into another edition of the same work.
  Which works have enough readers to recommend is decided in Section 5, on the sample.

In [5]:
# Build the book dimension from the four raw files (one to two minutes), or load the saved pickle.
books = bd.load_or_build_book_dimension(RAW, PROCESSED / "book_dimension.pkl", rebuild=REBUILD_BOOK_DIMENSION)
print(f"{len(books):,} works x {books.shape[1]} columns")
bd.describe_schema(books)        # the columns the three notebooks read; the rest are listed in the last row

loading /Users/davidhook/school/milestone1/data/processed/book_dimension.pkl


1,521,963 works x 34 columns


,dtype,example,used for
column,,,
work_id,int32,2792775,"key; joins to interactions, similar_books and the Open Library match (Books 1-3)"
canonical_book_id,int32,2767052,Goodreads edition id of the most-rated edition; the Open Library join key (Book 1)
title,object,"The Hunger Games (The Hunger Games, #1)",shown in explorer tooltips and recommendation lists (Books 1-3)
author_names,object,['Suzanne Collins'],list; first author shown with each title (Books 1-3)
author_ids,object,[153394],list; the same-author filter on recommendations (Book 2)
series_ids,object,[339872],list; the same-series filter on recommendations (Book 2)
series_names,object,['The Hunger Games'],list; shown when a recommendation is in a series (Books 2-3)
pub_year,int16,2008,"canonical edition's year, signed (BC negative); decade view, recency checks (Books 1-2)"
avg_rating,float32,4.34,ratings-weighted mean across editions; compared with Open Library ratings (Book 1)


### 2.3 What the collapse changed

The check that matters is the ratings total: every rating counted in the source is still counted
after the collapse. Rows fall by a third and memory by about a third; no evidence is lost.

In [6]:
# Before/after accounting for the edition -> work collapse.
bd.collapse_summary(books)

,source (editions),book dimension (works),note
measure,,,
rows,"2,360,655","1,521,963",35.5% fewer rows
columns,29 (+10 genre weights),34,"7 dropped, lists kept, genres flattened"
ratings counted,"958,938,616","958,938,616",preserved exactly
in memory,2.34 GB,1.49 GB,


### 2.4 Ratings are concentrated on few works

Visualization 2 ranks the 1.5 million works by number of ratings and plots the cumulative share of
all ratings. The top 1% of works hold about two thirds of all ratings; the top 10% hold over 90%.
Most of the catalogue has almost no ratings at all. This is the reason Section 5 needs a minimum
amount of evidence before a work can be recommended.

In [7]:
# Visualization 2: cumulative share of ratings held by the most-rated works.
cc.concentration_curve(books)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 3 · UCSD interactions: a 50,000-reader sample at work level

### 3.1 Decision: sample readers, not rows, and take 50,000 of them

The interactions file has 228.6 million rows and does not fit in memory on a laptop or in Colab.
Olivia's cleaning notebook established the approach: **sample readers and keep every row each
sampled reader has.** Sampling rows instead would break the links between books inside a reader's
history, which is what a co-reading recommender uses.

To choose the number of readers we measured five sample sizes on three things: the number of
works with enough evidence to recommend, how many of Goodreads' own "readers also enjoyed" pairs
survive (the evaluation benchmark used in Book 2), and how long an item-item model takes to build.

| Readers | All rows | Liked rows (4–5★) | Works with 10+ likes | Benchmark books usable | Build time |
|---|---|---|---|---|---|
| 10,000 | 2.6M | 0.83M | 11,987 | 8,549 | 0.1 min |
| 25,000 | 6.5M | 2.08M | 28,631 | 23,470 | 0.4 min |
| **50,000** | **13.0M** | **4.12M** | **52,356** | **46,475** | **1.1 min** |
| 100,000 | 25.7M | 8.27M | 91,847 | 85,273 | 2.3 min |

Build time was not a constraint at any size. 50,000 readers (5.7% of 876,145) gives four times the
catalogue of the 10,000-reader draft, keeps every model under two minutes to build, and leaves
about 5,000 readers each for validation and test after the split in Section 4. Going to 100,000
doubled the file for a small gain in benchmark coverage (5.4 to 6.6 Goodreads matches per book).

### 3.2 Decision: keep `is_read` and the full rating

The two model families in Book 2 read different signals from the same table. The shelf-presence
model uses every row; the directional and Jaccard models use likes (ratings of 4 or 5); and the
1–2 star rows are kept for the "which books did readers not enjoy" question raised in the proposal
review. Keeping the four columns `user_id, work_id, is_read, rating` serves all three.

### 3.3 Building the sample

Readers are chosen by a seeded permutation of the 876,145 reader ids, so the same readers are drawn
every time `SEED` and `SAMPLE_READERS` are unchanged. The CSV is streamed in 5-million-row chunks
and only sampled readers' rows are kept. The interactions file identifies editions by its own
index; `book_id_map.csv` converts that to a Goodreads edition id and the metadata converts the
edition to a work. A reader who shelved two editions of one work then has two rows; we keep one,
with the higher rating and `is_read = 1` if either copy was read. Editions with no work id in the
metadata (a few hundred rows) are dropped.

In [8]:
# Build the sample (about a minute: one pass over the 4.3 GB CSV, then the edition -> work map), or load it.
interactions = si.load_or_build_interactions(RAW, PROCESSED / "interactions_sample_50k.pkl",
                                             n_users=SAMPLE_READERS, seed=SEED, rebuild=REBUILD_INTERACTIONS)
si.write_submission_sample(interactions, PROJECT / "book1" / "data" / "interactions_sample_50k.first100.csv")
si.describe_interactions(interactions)

loading /Users/davidhook/school/milestone1/data/processed/interactions_sample_50k.pkl


readers                            50000
works                             797691
reader-work rows                12828893
rows marked read                 6393697
rows with a rating               5952086
likes (4-5 stars)                4133247
works with at least one like      453018
dtype: int64

### 3.4 Checks on the sampled rows

Three checks were run on the sampled rows before the edition-to-work step: missing keys, duplicate
reader-edition pairs, and ratings outside 0–5. A rating of 0 in this file means "shelved but not
rated", not missing; it is kept, and it is never counted as a like or a dislike.

In [9]:
# The three pre-filter checks, on the sampled readers' raw rows.
sampled_users = si.sample_user_ids(RAW / "user_id_map.csv", n_users=SAMPLE_READERS, seed=SEED)
raw_rows = si.read_sampled_interactions(RAW / "goodreads_interactions.csv", sampled_users)
si.quality_checks(raw_rows)

,found,action
check,,
"rows missing user_id, book_id or rating",0,none needed
duplicate reader-edition pairs,0,none needed
ratings outside 0-5,0,none needed
rating 0 = shelved but not rated,6937592,kept (53.7% of rows); excluded only from 'likes'


## 4 · Holding readers out for testing

### Decision: split by reader, 80/10/10, before anything is tuned

A model that has seen a reader's full history can reproduce that history, so the held-out data must
be readers the models never saw. Each sampled reader is assigned to **train (80%)**, **validation
(10%)** or **test (10%)** by hashing the reader id together with the seed. Hashing means the
assignment does not depend on row order and a reader keeps the same split if the sample is redrawn
with more readers.

Two rules follow from this and apply to the rest of the project:

- **Thresholds are computed on train only.** The minimum-likes floor in Section 5 counts likes among
  training readers, then is applied to validation and test.
- **Book 2 tunes on validation and reports on test once.** Book 3 tunes its fusion weights the same way.

In [10]:
# Assign every sampled reader to a split (saved for Books 2 and 3) and summarise the three parts.
assignment = tts.load_or_build_user_splits(interactions["user_id"].unique(), PROCESSED / "user_splits.pkl", seed=SEED)
splits = tts.split_interactions(interactions, assignment)
tts.summarise_split(splits)

loading /Users/davidhook/school/milestone1/data/processed/user_splits.pkl


,readers,reader-work rows,likes (4-5 stars),works with a like,share of readers
split,,,,,
train,39925,10142594,3292381,404249,0.798
validation,5007,1344528,418832,124218,0.100
test,5068,1341771,422034,121760,0.101


## 5 · How much evidence does each book have?

Before deciding which works a model may recommend, we need to see how much evidence each one has:
how many training readers liked it, and how consistent that approval is. The interactive explorer in
Section 8 shows the full picture, with the floor adjustable. The findings below are read from it, and
the table that follows records the decision in numbers.

### 5.1 Findings and the decision

1. **Half of all liked works have exactly one like** (50.6% on training readers). A recommendation
   drawn from such a work would be one reader's other favourites, not a pattern across readers.
2. **Approval is not informative at the low end.** With one or two raters the share who liked a work
   can only be 0%, 50% or 100%, which is why the left of the plot is a few horizontal bands and the
   right is a cloud centred near 75–85%.
3. **A low floor removes most works but few likes.** At 10 likes, 10.7% of liked works remain but
   76.7% of all training likes do, and 14.6% of the Goodreads benchmark books stay testable. Raising
   the floor further removes catalogue faster than it adds evidence per book.

**Decision: a work is recommendable if at least 10 training readers liked it.** Olivia proposed this
floor in her cleaning notebook after comparing 2, 5, 10 and 20 on her 10,000-reader sample; the
explorer confirms it on the larger sample and shows the trade-off continuously. The floor is stored
as training like-counts, so Book 2 can test other values on validation data without changing this
notebook.

In [11]:
# Count likes per work on training readers (the floor is applied from these counts) and show the effect per split.
train_works = ex.summarise_works(splits["train"], books)    # one row per work liked by a training reader
likes_per_work = tts.count_likes_per_work(splits["train"])
likes_per_work.to_pickle(PROCESSED / "train_likes_per_work.pkl")
tts.floor_report(splits, likes_per_work, floor=MIN_LIKES)

,works with a like,works at 10+ train likes,likes,likes kept,share of likes kept
split,,,,,
train,404249,43203,3292381,2526690,0.767
validation,124218,37665,418832,315228,0.753
test,121760,37830,422034,320311,0.759


## 6 · Open Library: the second source, two access methods

### 6.1 The API: one book at a time

Sarah wrote the project's Open Library lookup, `get_book_record()` in `scripts/open_library.py`.
Given an ISBN, a title, or a title and author, it calls the ISBN or search endpoint, scores the
candidate records by how complete their metadata is, follows the best one to its work, fetches the
work's ratings and genre tags, and returns one row. It is the path the dashboard uses when a reader
types a book. Each lookup makes three to five requests, and Open Library allows 3 requests per second
to clients that identify themselves with a `User-Agent` and an email address (1 per second otherwise).

The notebook does not call the API. The cell below shows the function's signature and docstring; the
table shows the rows her three example lookups returned when she ran them.

| Lookup | Title | Author | First published | Genre | Open Library rating | Ratings | Goodreads id |
|---|---|---|---|---|---|---|---|
| `isbn="9780140328721"` | Fantastic Mr. Fox | OL34184A (author key, not resolved to a name) | — | Fantasy | 3.95 | 121 | 1507552 |
| `title="The Hobbit", author="J.R.R. Tolkien"` | The Hobbit | J.R.R. Tolkien | 1937 | (none returned) | 4.29 | 500 | — |
| `title="Everything Was Beautiful and Nothing Hurt"` | Everything Was Beautiful and Nothing Hurt | Ben Reeves | 2026 | (none returned) | 4.00 | 1 | — |

Two things in those rows shaped the join in 6.2. The title searches returned no ISBN, no page count
and no genre, because the search endpoint's default records do not include the fields the
completeness score looks for; and only the ISBN lookup returned a Goodreads identifier. For a catalogue
join we need a field that is present for most works and matches a field we already have, which is
the Goodreads id.

In [12]:
# Sarah's lookup function, as written (scripts/open_library.py, Section 1). Shown, not called.
import inspect
print(inspect.signature(ol.get_book_record))
print(inspect.getdoc(ol.get_book_record))

(title=None, author=None, isbn=None)
Fetch a book from OpenLibrary using ISBN, title, or title + author.
Resolves to the most recent edition, fetches community reviews via the
Works API, and returns a one-row pandas DataFrame.


**Why the catalogue is joined through the dumps.** Open Library's developer guidelines ask that
clients not make hundreds of single-book requests and use the monthly dumps for bulk access. The
training catalogue has 43,203 recommendable works, which at three or more requests per book is
roughly 10 hours at the permitted rate. The dumps are the route the guidelines recommend.

### 6.2 Decision: join on the Goodreads book id, through the dumps

The proposal planned to join on ISBN. The reviewer suggested the Goodreads id instead, and the data
supports that: only 60–67% of works have an ISBN in the metadata, while Open Library edition
records carry a `goodreads` identifier that matches UCSD's `book_id` directly. The scan reads the
12.6 GB editions dump once (about three minutes), keeps editions whose Goodreads id is one of our canonical edition ids,
follows them to their Open Library work, reads that work's subjects from the works dump, and
aggregates the ratings dump per work. The result is saved as a pickle; later runs load it.

In [13]:
# Scan the three dumps once (about four minutes), or load the saved match. Then attach it to the book dimension.
match = ol.load_or_build_open_library_match(books, OPEN_LIBRARY, PROCESSED / "open_library_match.pkl",
                                            rescan=RESCAN_OPEN_LIBRARY)
books_ol = ol.merge_open_library_match(books, match)
ol.match_report(books_ol, train_works, floor=MIN_LIKES)

loading /Users/davidhook/school/milestone1/data/processed/open_library_match.pkl


,works,matched to Open Library,match rate,with subjects,with Open Library ratings,median subjects per matched work
all works,1521963,395948,0.260,343200,142793,5.0
works liked by a training reader,404249,131380,0.325,118575,73818,6.0
works with 10+ training likes,43203,19761,0.457,18865,17578,11.0


### 6.3 What Open Library adds

Two things the UCSD data does not have:

- **Subjects.** UCSD gives each work at most ten coarse genre labels. Open Library has free-text
  subjects (*The Hobbit* has 93). Book 2 uses them to check whether recommendations stay on topic at a
  finer level than genre.
- **Ratings after 2017.** UCSD's ratings stop at the 2017 scrape. Open Library's are current, so the
  two communities' views of the same books can be compared.

Visualization 3 shows how far the join reaches: the match rate by how many training readers liked a
work. The comparison of the two communities' ratings is the bottom-right panel of the explorer in
Section 8, where it follows the floor and the genre you choose.

In [14]:
# Visualization 3: match rate by band of training likes.
cc.open_library_coverage_chart(books_ol, train_works)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 7 · Visualization 1: the evidence explorer

The explorer places every work liked by a **training** reader by how many of them liked it (x axis,
log scale) and what share of its raters gave it 4–5 stars (y axis). Each cell counts the works at that
position; darker cells hold more works. The other three panels follow the same selection.

- **Slider** (below the chart): sets the minimum number of likes. The number beside the slider is its
  position; the floor it selects is printed on the chart ("floor = 10 likes"). The rule moves, cells
  to its left fade, and the lower-left panel shows what that floor keeps: share of likes, share of
  works, and share of Goodreads benchmark books still testable.
- **Hover** a cell to see how many works are there and the most-liked titles among them.
- **Click a genre** on the right to redraw every panel for that genre only; double-click to clear.
- **Type part of a title** to mark where particular books fall.
- **Bottom right:** Goodreads' average rating (to 2017) against Open Library's (2026) for the works
  above the floor with at least five Open Library ratings; hover a dot for the book.

In [15]:
# Visualization 1. Training readers only, with the Open Library ratings from Section 6 attached.
train_works_ol = ex.summarise_works(splits["train"], books_ol)
similar_books = np.load(PROCESSED / "similar_books.npy")   # Goodreads' own "readers also enjoyed" links (Book 2's benchmark)
ex.build_evidence_explorer(train_works_ol, similar_books, sample_readers=int(assignment.eq("train").sum()))

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 8 · What Book 2 receives

| File | One row per | Columns | Rows | Used by |
|---|---|---|---|---|
| `data/processed/book_dimension.pkl` | work | 34 (see §2.2) | 1,521,963 | all books |
| `data/processed/interactions_sample_50k.pkl` | reader × work | `user_id, work_id, is_read, rating` | 12,828,893 | Book 2 models |
| `data/processed/user_splits.pkl` | reader | `user_id` (index), `split` | 50,000 | Books 2 and 3 |
| `data/processed/train_likes_per_work.pkl` | work liked in train | `work_id` (index), likes | 404,249 | the recommendable floor |
| `data/processed/open_library_match.pkl` | matched work | `work_id, canonical_book_id, ol_work_key, ol_edition_key, isbn_13, ol_title, ol_subjects, ol_subject_count, ol_first_publish_date, ol_rating_count, ol_rating_mean` | 399,245 | Book 2 topic and rating checks |
| `data/processed/similar_books.npy` | Goodreads link | `work_id, similar_work_id` | 6,020,867 | Book 2 benchmark |

Files over 10 MB are represented in the submission by their first 100 records (`data/samples/`); the
full files are produced by running this notebook against the downloads in Section 1, except
`similar_books.npy`, which holds the books file's `similar_books` field at work level and was built in
an earlier exploration notebook; it is copied into `data/processed/` alongside the others.